# پایلوت امن Qwen MoE — فقط لایهٔ صفر

این نوت‌بوک عمداً فقط یک لایه را آموزش می‌دهد؛ هیچ‌کدام از ۲۴ لایه را به‌صورت خودکار اجرا نمی‌کند. هدف، سنجش مسیر فنی و خطای بازسازی FFN قبل از صرف زمان برای آموزش کامل است.

**مهم:** موفقیت این پایلوت به معنی حفظ استدلال یا آماده بودن مدل نهایی نیست. دادهٔ جمع‌آوری‌شده برای smoke test است، نه آموزش نهایی. مدل فعلی مطابق تنظیمات فعلی ریپو `Qwen/Qwen3.5-2B` است؛ اگر تصمیم نهایی تو حتماً نسخهٔ Instruct است، پیش از آموزش ۲۴ لایه باید مدل هدف را یک‌دست کنیم.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, subprocess, sys
WORK = Path('/content/drive/MyDrive/qwen-moe-layer0-pilot')
WORK.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.5-2B'  # Must match the repository's current teacher configuration
print('Persistent output:', WORK)
print('Teacher model:', MODEL_ID)


In [ ]:
# Download the latest committed repository and install its runtime dependencies.
!git clone https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git /content/Qwen-MoE-0.4B-
%cd /content/Qwen-MoE-0.4B-
!pip -q install -U transformers accelerate safetensors datasets
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU پیدا نشد. در Colab از Runtime > Change runtime type، شتاب‌دهندهٔ GPU را انتخاب کن و دوباره اجرا کن.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## ۱) دریافت دادهٔ فارسی و جداسازی سندها

۶۴ سند واقعی از مجموعه‌های تعریف‌شده در اسکریپت ریپو دریافت می‌شود؛ متن ساختگی جایگزین نخواهد شد. اگر دریافت داده شکست خورد، خطا را نگه دار و گزارشش را بررسی کن. این داده فقط برای تست فنی است.


In [ ]:
!python scripts/collect_persian_corpus.py --count 64 --min-chars 160 --max-chars 1800 --max-scan 20000 --output /content/fa_pilot_corpus.txt --manifest-output /content/fa_pilot_corpus.manifest.json
!python scripts/split_persian_corpus.py --input /content/fa_pilot_corpus.txt --train-output /content/fa_pilot_train.txt --eval-output /content/fa_pilot_eval.txt --manifest-output /content/fa_pilot_split.manifest.json --eval-fraction 0.15 --seed 20261025
import shutil
for src in ['/content/fa_pilot_corpus.txt','/content/fa_pilot_corpus.manifest.json','/content/fa_pilot_train.txt','/content/fa_pilot_eval.txt','/content/fa_pilot_split.manifest.json']:
    shutil.copy2(src, WORK / Path(src).name)
print(json.loads((WORK / 'fa_pilot_split.manifest.json').read_text()) | {'train_sha256':'[omitted]','evaluation_sha256':'[omitted]'})


## ۲) آموزش فقط لایهٔ صفر

تنظیمات محافظه‌کارانه‌تر از نوت‌بوک تمام‌لایه است: حداکثر طول ۱۲۸، ۵ epoch و ۵۱۲ توکن برای warm-start روتر. خروجی روی Google Drive ذخیره می‌شود تا با قطع شدن runtime از بین نرود.

**این سلول را اجرا کن؛ سلول آموزش ۲۴ لایه در این نوت‌بوک وجود ندارد.**


In [ ]:
!python scripts/distill_all_qwen_ffn_layers.py --model Qwen/Qwen3.5-2B --text-file /content/fa_pilot_train.txt --output-dir "/content/drive/MyDrive/qwen-moe-layer0-pilot/checkpoints" --num-experts 32 --top-k 2 --epochs 5 --max-length 128 --batch-size 1 --train-batch-size 128 --router-pretrain-tokens 512 --balance-loss-weight 0.01 --eval-fraction 0.1 --seed 20261025 --device cuda --start-layer 0 --end-layer 0


In [ ]:
checkpoint_path = WORK / 'checkpoints' / 'layer_00_top2_32experts.pt'
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
keys = ['model_id','teacher_revision','layer','tokens_captured','train_tokens','eval_tokens','train_documents','eval_documents','eval_mse','eval_relative_error','eval_cosine_similarity','dead_experts_on_eval','router_entropy_nats','balance_loss_weight']
for key in keys:
    print(f'{key}: {checkpoint.get(key)}')
print('Expert route assignments:', checkpoint.get('expert_route_assignments'))
print('Checkpoint saved to:', checkpoint_path)
print('\nتفسیر: این معیارها خطای بازسازی خروجی FFN لایهٔ صفر را می‌سنجند؛ حفظ توانایی استدلال، کیفیت گفت‌وگو یا سرعت موبایل را ثابت نمی‌کنند.')


## ۳) تصمیم پس از پایلوت

- اگر خطای نسبی بالا باشد، شباهت کسینوسی پایین باشد یا چندین expert بلااستفاده بمانند، **آموزش ۲۴ لایه را شروع نکن**؛ ابتدا روش Router/تقطیر باید اصلاح شود.
- حتی معیارهای خوب لایهٔ صفر کافی نیستند: باید ارزیابی مستقل سرتاسری و آزمون‌های فارسی/استدلال انجام شود.
- این خروجی فقط checkpoint پژوهشی یک لایه است؛ نه مدل کامل Transformers، نه GGUF و نه مدل آمادهٔ Android.
- گزارش‌ها و checkpoint در `MyDrive/qwen-moe-layer0-pilot/` باقی می‌مانند.
